In [0]:
# Leitura do arquivo CSV armazenado no Volume Raw.
# O cabeçalho é utilizado como nome das colunas e o Spark infere os tipos de dados.
df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .option("sep", ",") \
    .csv("/Volumes/salary_mvp/salary/raw/salary.csv")

In [0]:
# Visualização inicial dos dados após a ingestão,
# permitindo verificar a estrutura e o conteúdo do arquivo.
display(df)

In [0]:
# Verificação da quantidade de registros e colunas
# carregados a partir do arquivo original.
print("Linhas:", df.count())
print("Colunas:", len(df.columns))

In [0]:
# Listagem das colunas presentes no conjunto de dados.
print(df.columns)

In [0]:
# Verificação dos tipos de dados inferidos pelo Spark para cada coluna.
df.printSchema()

In [0]:
# Verificação da quantidade de valores nulos em cada coluna.
# Essa análise faz parte do diagnóstico inicial da qualidade dos dados.
from pyspark.sql.functions import col, sum, when

df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
]).show()

In [0]:
# Identificação de valores ausentes representados pelo caractere "?".
# Diferentemente de NULL, esses valores estão armazenados como texto.
from pyspark.sql.functions import trim

for c in df.columns:
    qtd = df.filter(trim(col(c)) == "?").count()
    if qtd > 0:
        print(c, "->", qtd)

In [0]:
# Verificação de registros duplicados considerando todas as colunas.
# A comparação entre o total original e o total após dropDuplicates()
# permite identificar a quantidade de registros repetidos.
print("Total de linhas:", df.count())
print("Linhas após remoção de duplicadas:", df.dropDuplicates().count())
print("Linhas duplicadas:", df.count() - df.dropDuplicates().count())

In [0]:
# Verificação da distribuição da variável salary,
# que será utilizada como referência nas análises de negócio.
display(
    df.groupBy("salary")
      .count()
      .orderBy("salary")
)

In [0]:
# Persistência dos dados brutos na camada Bronze.
# Nesta etapa, os dados são armazenados sem aplicação de regras de limpeza ou transformação.

df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("salary_mvp.salary.bronze_salary")

In [0]:
# Consulta da tabela Bronze para validar a persistência dos dados.
display(
    spark.table("salary_mvp.salary.bronze_salary")
)

In [0]:
# Validação da quantidade de registros armazenados na camada Bronze.
bronze_df = spark.table("salary_mvp.salary.bronze_salary")

print("Linhas na Bronze:", bronze_df.count())
print("Colunas na Bronze:", len(bronze_df.columns))

In [0]:
# Leitura dos dados armazenados na camada Bronze.
# A Silver será construída a partir dos dados persistidos nessa camada.

bronze_df = spark.table("salary_mvp.salary.bronze_salary")

In [0]:
# Criação da camada Silver a partir da Bronze.
# Nesta etapa, os valores ausentes representados por "?"
# são convertidos para NULL e os textos são padronizados.

from pyspark.sql.functions import col, trim, when

silver_df = bronze_df

# Substitui "?" por NULL nas colunas que possuem valores ausentes.
for c in ["workclass", "occupation", "native-country"]:
    silver_df = silver_df.withColumn(
        c,
        when(trim(col(c)) == "?", None).otherwise(trim(col(c)))
    )